## Download the exercise data
Run the next cell once before starting the exercise. It downloads and extracts this notebook’s data into `~/kenya2026`. Set `KENYA2026_WORK_DIR` first if you prefer another location.

In [ ]:
from pathlib import Path
import os
import subprocess

exercise = "day3_afternoon_gene_flow_dstat"
base_url = "https://popgen.dk/albrecht/course/kenya2026/data"
work_dir = Path(os.environ.get("KENYA2026_WORK_DIR", Path.home() / "kenya2026")).expanduser()
exercise_dir = work_dir / exercise
archive = work_dir / f"{exercise}.zip"
work_dir.mkdir(parents=True, exist_ok=True)

if not archive.exists():
    subprocess.run(["wget", "-c", f"{base_url}/{exercise}.zip", "-O", str(archive)], check=True)
if not exercise_dir.exists():
    subprocess.run(["unzip", "-q", str(archive), "-d", str(work_dir)], check=True)

os.chdir(exercise_dir)
print(f"Working directory: {Path.cwd()}")

In [ ]:
work_dir=${KENYA2026_WORK_DIR:-$HOME/kenya2026}
cd "$work_dir/day3_afternoon_gene_flow_dstat"
printf "Bash working directory: %s\n" "$PWD"

In [ ]:
work_dir <- Sys.getenv("KENYA2026_WORK_DIR", unset = file.path(path.expand("~"), "kenya2026"))
setwd(file.path(work_dir, "day3_afternoon_gene_flow_dstat"))
cat("R working directory:", getwd(), "\n")

### Software requirements
The setup cell above downloads **only the exercise data**. It does not install software. Before running the rest of this notebook, install the command-line programs and the Python or R packages that are imported or called in the exercises. If you see an error such as `command not found`, `ModuleNotFoundError`, or `there is no package called ...`, install the named dependency or ask an instructor for help.

# Gene flow 


## Objectives

1. What is the D statistic?
2. How to calculate the D statistic & interpret the results


We will read in some simulated data (that includes 4 populations), extract the genotype matrix, filter to biallelic sites & then calculate the D statistic

In [ ]:
library(vcfR)

# read in simulated vcf
vcf <- read.vcfR("hum_nea_siml.vcf.gz")
ingt <- extract.gt(vcf, element = "GT")

ingt[which(ingt=="0|0")]<-0
ingt[which(ingt=="0|1")]<-1
ingt[which(ingt=="1|0")]<-1
ingt[which(ingt=="1|1")]<-2

# how many loci
nrow(ingt)

#  unique(ingt[,1])
# [1] "0|0" "1|1" "1|0" "0|1" "2|0" "2|2" "1|2" "0|2" "1|3" "2|1"

# filter to biallelic sites only
biallelic <- ingt[!apply(ingt, 1, function(row) any(row %in% c("2|0", "2|2", "1|2", "0|2", "1|3", "2|1"))), ]

nrow(biallelic)

biallelic[which(biallelic=="0|0")]<-0
biallelic[which(biallelic=="0|1")]<-1
biallelic[which(biallelic=="1|0")]<-1
biallelic[which(biallelic=="1|1")]<-2

biallelicdf<-as.data.frame(biallelic)
# convert to numeric
biallelicdf[] <- lapply(biallelicdf, function(x) {if (is.character(x)) as.numeric(x) else x})

In [ ]:
# sample info, individuals of each population
sampleinfo <- read.table("hum_nea_siml.tsv", header = T, sep = "\t")

outg <- sampleinfo[which(sampleinfo$pop == "CHIMP"),1]
p1 <- sampleinfo[which(sampleinfo$pop == "AFR"),1]
p2 <- sampleinfo[which(sampleinfo$pop == "EUR"),1]
p3 <- sampleinfo[which(sampleinfo$pop == "NEA"),1]


In [ ]:
# function to calc d statistic
calc_abba_baba <- function(ingt1) {
  
  f1 <- rowMeans(ingt1[, p1, drop = FALSE], na.rm = TRUE) / 2
  f2 <- rowMeans(ingt1[, p2, drop = FALSE], na.rm = TRUE) / 2
  f3 <- rowMeans(ingt1[, p3, drop = FALSE], na.rm = TRUE) / 2
  fo <- rowMeans(ingt1[, outg, drop = FALSE], na.rm = TRUE) / 2
  
  keep <- is.finite(f1) & is.finite(f2) & is.finite(f3) & is.finite(fo)
  
  abba <- sum((1 - f1[keep]) * f2[keep] * f3[keep])
  baba <- sum(f1[keep] * (1 - f2[keep]) * f3[keep])
  
  D <- (abba - baba) / (abba + baba)
  
return(cbind(abba = abba,baba = baba,D = D,n_sites = sum(keep)))}

calc_abba_baba(ingt1=biallelicdf)


### Questions (1)
1. Please interpret the table you generated in the previous code block.
2. Does it fit with what we learnt in the lecture?
3. If this was your analysis what would be your next step?

### Questions (2)

4. What does the D statistic detect?

A. Evidence of introgression or gene flow between populations  
B. Genetic drift within a single population  
C. The mutation rate of a population  
D. Effective population size of a population  

5. Why do we need an outgroup for the D statistic?

A. It identifies which population has the largest effective population size  
B. It provides the ancestral/reference allele state  
C. It identifies the population with the most introgression  
D. It determines the mutation rate  

6. If the number of ABBA sites = 200 and number of BABA sites = 200, what is the value of D statistic?

A. -1  
B. -0.5  
C. 0  
D. 1  

7. If the number of ABBA sites = 600 and number of BABA sites = 400, what is the value of D statistic?

A. -0.20  
B. 0  
C. 0.40  
D. 0.20  
Hint: D = (ABBA - BABA) / (ABBA + BABA)

8. If P1 and P3 had gene flow which site pattern would be in excess?

A. ABBA  
B. BABA  
C. AABB  
D. Neither  

## F4 / D-statistic
Here we are going to use real data to understand gene flow in Wilderbeet populations.

For this part of the exercise we can use the R package admixtools to compute F4 values which correspond to the D-statistic mentioned today, just with a flipped sign, so that negative values of F4 correspond to a positive D-statistic and vice versa.

Here we take a look at whether any of the populations of blue wildebeest are more closely related to the black wildebeest than the others - which indicates gene flow.

In [ ]:
library(admixtools)
library(tidyverse)
options(repr.plot.width=16)

# load in f2 values that were pre computed from the plink data set
f2 <- read_f2("wildebeest_fstats_wildebeestref")

# define tree of relationships between groups, this format is a bit hard to read,
#  but is similar to newick format for those familiar
tree <- c(c(c(c(c(c('N-Selous', 'C-Luangwa'), 'B-Ethosha'), c(c(c("E-Nairobi", "E-Amboseli"), "E-Monduli"), 'W-Serengeti')), 'black'), "hartebeest"))

# generate f4 values
f4 <- f4(f2, tree, f4mode = FALSE)

# select only values for subtrees where pop3 is black wildebeest and outgroup is hartebeest
f4_sub <- f4[f4$pop4=="hartebeest" & f4$pop3=="black",]

# make new row for each combination of pop 1 and pop 2 where they are switched and their f4 flipped, to make plot look nicer
f4_sub_switched <- f4_sub %>%
                          mutate(
                            temp = .data[["pop1"]],
                            !!"pop1" := .data[["pop2"]],
                            !!"pop2" := temp,
                            across(all_of(c("est", "z" )), ~ . * -1)
                          ) %>%
                          select(-temp)

f4_sub <- bind_rows(f4_sub, f4_sub_switched)

# show resulting subset
f4_sub

In this output, look for values in the column labelled 'z'. Those rows that have values higher than 3 or lower than -3 are usually considered statistically significant. If the value is negative, it means that the population in column pop2 has more alleles in common with black wildebeest than the population in column pop1. If it's positive, the interpretation is the opposite, i.e. pop1 has more alleles in common with black than pop2.

This can also be shown in a plot. Run this plotting code and see if you can make sense of the plot below.

In [ ]:
ggplot(f4_sub, aes(est, pop1, fill = pop2)) +
geom_bar(stat = "identity", position = "dodge") +
facet_wrap(~pop1, ncol=1, scale="free_y") +
theme_bw() +
xlab("f4 value \n with population 3 as black wildebeest and population 4 as hartebeest") +
theme(axis.text=element_text(size=20), legend.text = element_text(size=30), axis.title=element_text(size=20),
      strip.background = element_blank(), strip.text.x = element_blank(), legend.key.size = unit(2, 'cm'), legend.title = element_text(size=30))

**Which population of blue wildebeest had gene flow with black wildebeest?  
How do you think this happened, and why is it not all blue wildebeests that had gene flow with black wildebeest?  
Considering what you learn today, why do use hartebeest on pop4?  
What would happen if we use another black wildebeest on pop4?**